# 🤖 AI Engineering Fundamentals — Lezione 5
## Notebook Gruppo C

**ITS Novitas 4.0 | Giovedì 04/06/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "C"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [1]:
!pip install anthropic requests -q
from google.colab import userdata
import anthropic, os, json, requests

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 19.0 MB/s eta 0:00:00
✅ Setup completato!


---
## 🎯 Tema del Gruppo C: Tool Use & Loop Agente

Esplorate come il modello decide quale tool chiamare
e come gestire il loop agente — incluso il caso multi-step
dove il modello chiama più tool in sequenza.

---
### Esercizio 1 — Vedere la decisione del modello *(guidato)*

Prima di implementare il loop completo, osservate cosa restituisce
il modello quando decide di usare un tool.
La risposta NON è testo — è JSON con la decisione.

In [3]:
# Esercizio 1 — osservare la decisione del modello

tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui un'operazione matematica precisa. Usa questo tool per qualsiasi calcolo aritmetico.",
    "input_schema": {
        "type": "object",
        "properties": {
            "espressione": {
                "type": "string",
                "description": "L'espressione matematica da calcolare. Es: '234 * 567'"
            }
        },
        "required": ["espressione"]
    }
}

# Chiamata CON tool — NON implementiamo il loop, solo osserviamo
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[tool_calcolatrice],
    messages=[{"role": "user", "content": "Quanto fa 1234 * 5678?"}]
)

print(f"stop_reason: {response.stop_reason}")
print(f"\nContenuto della risposta:")
for block in response.content:
    print(f"  tipo: {block.type}")
    if block.type == "tool_use":
        print(f"  nome tool: {block.name}")
        print(f"  parametri: {block.input}")
        print(f"  id: {block.id}")

print()
print("💡 Il modello NON ha calcolato nulla.")
print("   Ha solo detto: chiama 'calcola' con questa espressione.")
print("   Il calcolo lo fa Python — non Claude.")

stop_reason: tool_use

Contenuto della risposta:
  tipo: tool_use
  nome tool: calcola
  parametri: {'espressione': '1234 * 5678'}
  id: toolu_01JePeFgNJU8264bUo9m6NN3

💡 Il modello NON ha calcolato nulla.
   Ha solo detto: chiama 'calcola' con questa espressione.
   Il calcolo lo fa Python — non Claude.


---
### Esercizio 2 — Il loop completo *(guidato)*

Implementate il loop agente completo con il contatore
di sicurezza per evitare loop infiniti.

In [7]:
import anthropic # Re-import anthropic because the original cell might be re-executed independently
import os, json, requests # Re-import other necessary modules
from google.colab import userdata # Re-import userdata for API key

# Ensure ANTHROPIC_API_KEY is set if this cell is run independently
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# client = anthropic.Anthropic() # Re-initialize client if needed

# Esercizio 2 — tool loop completo

def calcola(espressione: str) -> str:
    """Calcolatrice sicura con lista bianca di caratteri."""
    allowed = set('0123456789+-*/().% ')
    if not all(c in allowed for c in espressione):
        return "Errore: espressione non valida"
    try:
        return f"{espressione} = {eval(espressione)}"
    except Exception as e:
        return f"Errore: {str(e)}"

def esegui_tool(nome, parametri):
    """Router: smista la chiamata al tool giusto."""
    if nome == "calcola":
        return calcola(parametri["espressione"])
    return f"Tool '{nome}' non trovato"

MAX_ITERAZIONI = 10

def chat_con_tool(messaggio, tools):
    """Chatbot con tool loop e contatore di sicurezza."""
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0

    while True:
        iterazioni += 1
        # TODO: aggiungete il controllo MAX_ITERAZIONI
        if iterazioni > MAX_ITERAZIONI:
            return "Errore: loop agente non terminato (superato MAX_ITERAZIONI)"

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # TODO: gestite end_turn — restituite il testo
        if response.stop_reason == "end_turn":
            # Check if content is not empty before attempting to access text
            if response.content:
                text_blocks = [b for b in response.content if b.type == "text"]
                if text_blocks:
                    testo = text_blocks[0].text
                else:
                    testo = "No text content in response."
            else:
                testo = "No content in response."
            history.append({"role": "assistant", "content": response.content})
            return testo, history

        # TODO: gestite tool_use — eseguite i tool e aggiungete i risultati
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ {risultato}")
                    # TODO: aggiungete il risultato a tool_results
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})

# Test
print("❓ Quanto fa 1234 * 5678?")
print(chat_con_tool("Quanto fa 1234 * 5678?", [tool_calcolatrice]))

❓ Quanto fa 1234 * 5678?
  🔧 calcola({'espressione': '1234 * 5678'})
  ✅ 1234 * 5678 = 7006652
('**1234 × 5678 = 7.006.652**', [{'role': 'user', 'content': 'Quanto fa 1234 * 5678?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_019GQVH6STzb9Dn8s65dzh2y', caller=DirectCaller(type='direct'), input={'espressione': '1234 * 5678'}, name='calcola', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_019GQVH6STzb9Dn8s65dzh2y', 'content': '1234 * 5678 = 7006652'}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='**1234 × 5678 = 7.006.652**', type='text')]}])


---
### Esercizio 3 — Multi-step: tool concatenati *(libero)*

Fate una domanda che richiede 2 tool in sequenza.
Osservate quante iterazioni fa il loop e in che ordine
vengono chiamati i tool.

In [11]:
# Esercizio 3 — multi-step con tool concatenati

# Tool Wikipedia per cercare informazioni
tool_wikipedia = {
    "name": "cerca_wikipedia",
    "description": "Cerca informazioni su Wikipedia. Usa per fatti, biografie, concetti tecnici.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Termine da cercare"}
        },
        "required": ["query"]
    }
}

def cerca_wikipedia(query: str) -> str:
    try:
        url = f"https://it.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            return r.json().get("extract", "")[:500]
        return f"Nessun risultato per '{query}'"
    except Exception as e:
        return f"Errore: {str(e)}"

# Aggiornate il router
def esegui_tool_v2(nome, parametri):
    if nome == "calcola":          return calcola(parametri["espressione"])
    if nome == "cerca_wikipedia":  return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    return f"Tool '{nome}' non trovato"

# TODO: aggiornate chat_con_tool per usare esegui_tool_v2

def chat_con_tool(messaggio, tools):
    """Chatbot con tool loop e contatore di sicurezza."""
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0

    while True:
        iterazioni += 1
        # TODO: aggiungete il controllo MAX_ITERAZIONI
        if iterazioni > MAX_ITERAZIONI:
            return "Errore: loop agente non terminato (superato MAX_ITERAZIONI)"

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # TODO: gestite end_turn — restituite il testo
        if response.stop_reason == "end_turn":
            # Check if content is not empty before attempting to access text
            if response.content:
                text_blocks = [b for b in response.content if b.type == "text"]
                if text_blocks:
                    testo = text_blocks[0].text
                else:
                    testo = "No text content in response."
            else:
                testo = "No content in response."
            history.append({"role": "assistant", "content": response.content})
            print(f"\n{iterazioni}iterazioni")
            return testo, history

        # TODO: gestite tool_use — eseguite i tool e aggiungete i risultati
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ {risultato}")
                    # TODO: aggiungete il risultato a tool_results
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})

# Poi testate con domande multi-step:

domande_multistep = [
    # Richiede Wikipedia + calcolatrice
    "Anthropic è stata fondata nel 2021. Quanti anni ha adesso (siamo nel 2026)?",
    # Richiede 2 ricerche Wikipedia
    "Chi ha fondato Anthropic e chi ha fondato OpenAI?",
    # La vostra domanda multi-step
    "Quanti abitanti ha Sassari e quanti ne ha Flussio e poi addizionali",  # ← inventatene una voi
]

TUTTI_I_TOOL = [tool_calcolatrice, tool_wikipedia]

for domanda in domande_multistep:
    if domanda:
        print(f"\n{'='*55}")
        print(f"❓ {domanda}")
        print(chat_con_tool(domanda, TUTTI_I_TOOL))

        # TODO: chiamate chat_con_tool con TUTTI_I_TOOL
        # Contate quante iterazioni ha fatto il loop

# Conclusione: quante iterazioni per ogni domanda?
# Il modello chiama i tool nell'ordine che vi aspettavate?
# ...


❓ Anthropic è stata fondata nel 2021. Quanti anni ha adesso (siamo nel 2026)?
  🔧 calcola({'espressione': '2026 - 2021'})
  ✅ 2026 - 2021 = 5

2iterazioni
('Anthropic ha **5 anni** nel 2026 (essendo stata fondata nel 2021).', [{'role': 'user', 'content': 'Anthropic è stata fondata nel 2021. Quanti anni ha adesso (siamo nel 2026)?'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='Per calcolare quanti anni ha Anthropic adesso:', type='text'), ToolUseBlock(id='toolu_017BZ44D1ttPKfM9SSoNPZTH', caller=DirectCaller(type='direct'), input={'espressione': '2026 - 2021'}, name='calcola', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_017BZ44D1ttPKfM9SSoNPZTH', 'content': '2026 - 2021 = 5'}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='Anthropic ha **5 anni** nel 2026 (essendo stata fondata nel 2021).', type='text')]}])

❓ Chi ha fondato Anthropic e chi ha fondato OpenAI?
  🔧 cerca_wikipedia({'query': 'Anth

---
### Esercizio 4 — La description conta *(libero)*

Cambiate la description del tool calcolatrice in modo vago.
Il modello lo usa ancora correttamente?
Dimostrate che la description è la parte più critica della definizione.

In [19]:
# Esercizio 4 — l'impatto della description

tool_vago = {
    "name": "calcola",
    "description": "Tool.",  # ← description completamente vaga
    "input_schema": tool_calcolatrice["input_schema"]
}

tool_fuorviante = {
    "name": "calcola",
    "description": "Usa questo tool per rispondere a domande generali.",  # ← fuorviante
    "input_schema": tool_calcolatrice["input_schema"]
}

domande_test = [
    "Quanto fa 15% di 847?",
    "Qual è la radice quadrata di 144?",
    "Fammi una poesia sulla Sardegna.",  # non dovrebbe usare la calcolatrice
]

def esegui_tool_v2(nome, parametri):
    if nome == "calcola":          return calcola(parametri["espressione"])
    return f"Tool '{nome}' non trovato"

def chat_con_tool(messaggio, tools):
    """Chatbot con tool loop e contatore di sicurezza."""
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0

    while True:
        iterazioni += 1
        # TODO: aggiungete il controllo MAX_ITERAZIONI
        if iterazioni > MAX_ITERAZIONI:
            return "Errore: loop agente non terminato (superato MAX_ITERAZIONI)"

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # TODO: gestite end_turn — restituite il testo
        if response.stop_reason == "end_turn":
            # Check if content is not empty before attempting to access text
            if response.content:
                text_blocks = [b for b in response.content if b.type == "text"]
                if text_blocks:
                    testo = text_blocks[0].text
                else:
                    testo = "No text content in response."
            else:
                testo = "No content in response."
            history.append({"role": "assistant", "content": response.content})
            print(f"\n{iterazioni}iterazioni")
            return testo, history

        # TODO: gestite tool_use — eseguite i tool e aggiungete i risultati
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ {risultato}")
                    # TODO: aggiungete il risultato a tool_results
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})
for domanda in domande_test:
    print(f"\n{'='*55}")
    print(f"❓ {domanda}")
    print(chat_con_tool(domanda, [tool_vago]))
    print(chat_con_tool(domanda, [tool_fuorviante]))

# TODO: testate le stesse domande con i 3 tool:
# tool_calcolatrice (description precisa)
# tool_vago (description vuota)
# tool_fuorviante (description sbagliata)

# Il modello si comporta diversamente?
# Quando usa il tool e quando no?

# Conclusione:
# La description impatta il comportamento? Sì/No e come?
# Regola pratica che dareste a chi costruisce tool:
# ...


❓ Quanto fa 15% di 847?
  🔧 calcola({'espressione': '847 * 0.15'})
  ✅ 847 * 0.15 = 127.05

2iterazioni
('Il 15% di 847 è **127,05**.', [{'role': 'user', 'content': 'Quanto fa 15% di 847?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_014tRXj8gpQ7o8MxgAKvCeuB', caller=DirectCaller(type='direct'), input={'espressione': '847 * 0.15'}, name='calcola', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_014tRXj8gpQ7o8MxgAKvCeuB', 'content': '847 * 0.15 = 127.05'}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='Il 15% di 847 è **127,05**.', type='text')]}])
  🔧 calcola({'espressione': '847 * 0.15'})
  ✅ 847 * 0.15 = 127.05

2iterazioni
('Il **15% di 847 è 127,05**.', [{'role': 'user', 'content': 'Quanto fa 15% di 847?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01SnM9bozXmgsrV7FcRixE7B', caller=DirectCaller(type='direct'), input={'espressione': '847 * 0.15'}, name='calcola', type='tool_use')]}, {'r

---
## 📊 Preparate la presentazione (5 slide)

1. **Il modello decide, Python esegue** — mostrate il JSON della decisione
2. **Il tool loop** — le 3 fasi: aggiungi assistant, esegui tool, aggiungi risultati
3. **Multi-step** — quante iterazioni per ogni domanda?
4. **La description conta** — risultati del confronto vago/preciso/fuorviante
5. **La vostra regola pratica** — come scrivere una buona description

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*